# 537 — Sports pool / promotion simulator (live EDA)

**Typical session:** open **`sports/sim_config.py`** and this notebook side by side. Run **Cell 0** (optional), **Cell 1** (helpers), then **Cell 10** (widgets + plot). See **`sports/documents/537_Manual.md`** for every knob, binning mode, and symbol.

Backup of the notebook **before batch cells were removed** (CELL 2–9 + archive banner): **`obsolete_files/sports_gameplan_old/537_Sports_Simulation_pre_archive_8MAy.ipynb`**.


In [5]:
### EDIT BLANK · session log

 #Burn cell for `.cursor/rules` diff sync. Not a numbered CELL.

# - Session marker: 2026-05-08 — Operator manual → `sports/documents/537_Manual.md`.
# - Session marker: 2026-05-08 — CELL 10: PNG via BytesIO + IPython.Image in widget Output (no duplicate plots); annotate_simulation_context: 9pt, left, tighter y.
# - Session marker: 2026-05-08 — CELL 10: display(fig)+close(fig) instead of plt.show() (avoid duplicate inline plots).
# - Session marker: 2026-05-08 — Slim notebook: CELL 2–9 removed; backup → obsolete_files/sports_gameplan_old/537_Sports_Simulation_pre_archive_8MAy.ipynb.
# - Session marker: 2026-05-08 — equal-width pool bins; archive banner; 537_Manual; drop Cell 8a preset.
# - Session marker: 2026-05-08 — pool-mean-A binning (N_POOLS vs agg bins) + EDA binning mode in Cell 10.
# - Session marker: 2026-05-08 — Cell 10 single-panel EDA + FIG_PLAYGROUND_EDA_INCHES; ability & winner dropdowns.
# - Session marker: 2026-05-08 — standardized plot metadata block (Selection + full sim lineage).
# - Session marker: 2026-05-08 — fig sizes ~square panels + CELL 10: global _playground_ready (fix nonlocal SyntaxError).
# - Session marker: 2026-05-08 — CELL 10: wire batch lock + bins into UI; Cell 8a preset; avoid duplicate nonlocal.
# - Session marker: 2026-05-08 — CELL 10: batch-scale lock + bins slider + Cell 8a preset (sawtooth).
# - Session marker: 2026-05-08 — CELL 10: left = no-pool global-rank ref; center/right = pools + widget score; right x=global rank.
# - Session marker: 2026-05-08 — simulation context captions below axes (not upper legend box).
# - Session marker: 2026-05-08 — CELL 10: fix suptitle f-strings; disable additive w when local-rank-only.
# - Session marker: 2026-05-07 — initial 537 scaffold.
# - Session marker: 2026-05-07 — first-pass simulations 1-3.
# - Session marker: 2026-05-07 — reset promotion weight rules to defaults.
# - Session marker: 2026-05-07 — align plots to four diagnostic questions.
# - Session marker: 2026-05-07 — add optional convergence plots.
# - Session marker: 2026-05-07 — annotate plots with A_i distribution.
# - Session marker: 2026-05-07 — render A_i labels with mathtext.
# - Session marker: 2026-05-07 — annotate plots with pool sorting mode.
# - Session marker: 2026-05-07 — implement pure assortative pool option.
# - Session marker: 2026-05-07 — restructure plots into six requested diagnostics.
# - Session marker: 2026-05-07 — extend Cell 6 to full rollup progression.
# - Session marker: 2026-05-08 — add Plot 7a/7b: global rank x-axis under local-rank promotion.
# - Session marker: 2026-05-08 — add noisy sorting and 50/50 additive blend.
# - Session marker: 2026-05-08 — pause convergence and add standalone plot context.
# - Session marker: 2026-05-08 — add plot 7: local-rank promotions ordered by global rank.
# - Session marker: 2026-05-08 — move simulation choices into sports/sim_config.py.
# - Session marker: 2026-05-08 — add reload_sim_config calls to runnable cells.
# - Session marker: 2026-05-08 — Cell 10 interactive playground (ipywidgets).

### CELL 0 — Config · A/B/C choices

This cell makes the first modeling forks explicit. For the first pass we implement the A choices and leave the B/C choices visible as planned extensions.

In [6]:
# CELL 0 — Load simulation config

from pathlib import Path
import importlib.util

CONFIG_CANDIDATES = [Path("sports/sim_config.py"), Path("sim_config.py")]


def reload_sim_config(verbose: bool = True):
    """Reload sports/sim_config.py and refresh all uppercase config variables."""
    sim_config_path = next((path for path in CONFIG_CANDIDATES if path.exists()), None)
    if sim_config_path is None:
        raise FileNotFoundError("Could not find sim_config.py in ./sports/ or current directory.")

    spec = importlib.util.spec_from_file_location("sim_config", sim_config_path)
    sim_config = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(sim_config)

    for name in dir(sim_config):
        if name.isupper():
            globals()[name] = getattr(sim_config, name)

    globals()["SIM_CONFIG_PATH"] = sim_config_path
    globals()["sim_config"] = sim_config

    if verbose:
        print("Reloaded simulation config from", sim_config_path)
        print(
            f"N={N_INDIVIDUALS}, K={N_WINNERS}, pools={N_POOLS}, runs={N_RUNS}, "
            f"ability_choice={ABILITY_DISTRIBUTION_CHOICE!r}, winner_choice={WINNER_SELECTION_CHOICE!r}, "
            f"pool_choices={LOCAL_POOL_ASSIGNMENT_CHOICES_SIM3}, sorting_noise_sd={SORTING_NOISE_SD}, "
            f"additive_w={ADDITIVE_LOCAL_RANK_WEIGHT}, convergence={SHOW_CONVERGENCE_PLOTS}"
        )
    return sim_config


reload_sim_config()
print("CELL 0 ready")


Reloaded simulation config from sim_config.py
N=1000, K=50, pools=50, runs=500, ability_choice='B', winner_choice='A', pool_choices=('A', 'B'), sorting_noise_sd=0.01, additive_w=0.0, convergence=False
CELL 0 ready


### CELL 1 — Imports and simulation helpers

These helpers keep the simulations comparable. Each scenario produces individual-run rows with ability, rank, optional pool fields, and a promoted flag.

In [7]:
# CELL 1 — Imports and simulation helpers

reload_sim_config()

if RUN_CELL1:
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import display

    # Inches per subplot row/column so multi-panel figures have nearly square axes.
    FIG_PANEL_IN = 5.8

    def figsize_grid(nrows: int, ncols: int) -> tuple[float, float]:
        """Figure (width, height) for an nrows×ncols grid of nearly square subplots."""
        p = FIG_PANEL_IN
        return (p * ncols, p * nrows)

    def generate_ability(rng: np.random.Generator, n: int, choice: str = "A") -> np.ndarray:
        """Generate individual ability A_i for one mock population."""
        if choice == "A":
            return rng.uniform(0.0, 1.0, size=n)
        if choice == "B":
            vals = rng.normal(loc=0.5, scale=0.18, size=n)
            return np.clip(vals, 0.0, 1.0)
        if choice == "C":
            vals = rng.beta(a=2.0, b=5.0, size=n)
            return vals / vals.max() if vals.max() > 0 else vals
        raise ValueError(f"Unknown ability distribution choice: {choice!r}")

    def rank_score(values: np.ndarray) -> np.ndarray:
        """Percentile-like rank score in (0, 1], where larger values are better."""
        s = pd.Series(values)
        return s.rank(method="average", pct=True).to_numpy(dtype=float)

    def normalized_weights(score: np.ndarray) -> np.ndarray:
        """Convert nonnegative scores into a probability vector."""
        w = np.asarray(score, dtype=float)
        w = np.where(np.isfinite(w), w, 0.0)
        w = np.clip(w, 0.0, None)
        total = float(w.sum())
        if total <= 0:
            return np.full(len(w), 1.0 / len(w))
        return w / total

    def choose_winners(
        rng: np.random.Generator,
        weights: np.ndarray,
        k: int,
        choice: str = "A",
    ) -> np.ndarray:
        """Return a boolean winner vector for one run."""
        n = len(weights)
        if choice == "A":
            p = normalized_weights(weights)
            idx = rng.choice(n, size=k, replace=False, p=p)
            winners = np.zeros(n, dtype=bool)
            winners[idx] = True
            return winners
        if choice == "B":
            p = np.minimum(normalized_weights(weights) * k, 1.0)
            return rng.uniform(size=n) < p
        if choice == "C":
            idx = np.argsort(weights)[-k:]
            winners = np.zeros(n, dtype=bool)
            winners[idx] = True
            return winners
        raise ValueError(f"Unknown winner selection choice: {choice!r}")

    def equal_pool_template(n: int, n_pools: int) -> np.ndarray:
        """Pool labels with approximately equal pool sizes."""
        return np.repeat(np.arange(n_pools), int(np.ceil(n / n_pools)))[:n]

    def sorting_signal_for_pools(
        rng: np.random.Generator,
        ability: np.ndarray,
        sorting_noise_sd: float = 0.0,
    ) -> np.ndarray:
        """Noisy signal used only for pool assignment; true A_i is unchanged."""
        noise_sd = max(float(sorting_noise_sd), 0.0)
        if noise_sd == 0.0:
            return np.asarray(ability, dtype=float)
        return np.asarray(ability, dtype=float) + rng.normal(loc=0.0, scale=noise_sd, size=len(ability))

    def assign_pool_ids(
        rng: np.random.Generator,
        ability: np.ndarray,
        n_pools: int,
        choice: str = "A",
        sorting_noise_sd: float = 0.0,
    ) -> tuple[np.ndarray, np.ndarray]:
        """Assign individuals to pools under random, assortative, or disassortative sorting."""
        n = len(ability)
        base = equal_pool_template(n, n_pools)
        pool_id = np.empty(n, dtype=int)
        signal = sorting_signal_for_pools(rng, ability, sorting_noise_sd)
        if choice == "A":
            return rng.permutation(base), signal
        order = np.argsort(signal)
        if choice == "B":
            # Assortative: adjacent sorting-signal ranks are grouped into the same pool.
            pool_id[order] = base
            return pool_id, signal
        if choice == "C":
            # Disassortative: adjacent sorting-signal ranks are spread across pools.
            pool_id[order] = np.arange(n) % n_pools
            return pool_id, signal
        raise ValueError(f"Unknown local pool assignment choice: {choice!r}")

    def add_pool_fields(df: pd.DataFrame) -> pd.DataFrame:
        """Add pool quality and local rank fields for rows with a pool_id column."""
        out = df.copy()
        out["pool_quality"] = out.groupby(["run", "pool_id"], observed=True)["A"].transform("mean")
        out["local_rank_score"] = out.groupby(["run", "pool_id"], observed=True)["A"].rank(method="average", pct=True)
        return out

    def summarize_by_bins(
        df: pd.DataFrame,
        x_col: str,
        n_bins: int = 10,
        label: str | None = None,
    ) -> pd.DataFrame:
        """Bin x_col and summarize observed promotion probability."""
        work = df.dropna(subset=[x_col, "promoted"]).copy()
        work["bin"] = pd.qcut(work[x_col], q=n_bins, labels=False, duplicates="drop")
        summary = (
            work.dropna(subset=["bin"])
            .groupby("bin", observed=True)
            .agg(
                x_mean=(x_col, "mean"),
                promotion_rate=("promoted", "mean"),
                n=("promoted", "size"),
            )
            .reset_index()
        )
        summary["x_col"] = label or x_col
        return summary

    def summarize_by_pool_mean_A_bins(
        df: pd.DataFrame,
        n_pool_bins: int,
        *,
        method: str = "equal_count",
        label: str | None = None,
    ) -> pd.DataFrame:
        """Bin *pools* by mean A_i, then average promotion within each bin.

        ``method``:
        - ``equal_count``: sort pools by mean A; ``numpy.array_split`` into ``K`` groups
          (as equal pool counts as possible). Needs ``K <=`` pools per run.
        - ``equal_width``: within each run, ``K`` equal-width intervals on
          ``[min mean A, max mean A]`` across pools (``pd.cut``). Bins can be empty;
          Q order follows increasing mean-A intervals.

        ``x_mean`` is mean individual A in the bin; rows sorted by ``x_mean`` (Q1..QK =
        low → high talent).
        """
        need = {"run", "pool_id", "A", "promoted"}
        missing = need - set(df.columns)
        if missing:
            raise ValueError(f"summarize_by_pool_mean_A_bins missing columns: {missing}")

        if method not in ("equal_count", "equal_width"):
            raise ValueError("method must be 'equal_count' or 'equal_width'")

        n_pool_bins = int(n_pool_bins)
        if n_pool_bins < 2:
            raise ValueError("n_pool_bins must be at least 2")

        work = df.dropna(subset=["A", "promoted", "pool_id"]).copy()
        work["pool_id"] = work["pool_id"].astype(np.int64)

        def pool_bin_map_for_run(sub: pd.DataFrame) -> pd.DataFrame:
            run_id = int(sub["run"].iloc[0])
            pool_mean = sub.groupby("pool_id", observed=True)["A"].mean()
            n_pools_run = int(pool_mean.shape[0])

            if method == "equal_count":
                pool_mean = pool_mean.sort_values(kind="mergesort")
                if n_pool_bins > n_pools_run:
                    raise ValueError(
                        f"n_pool_bins ({n_pool_bins}) exceeds number of pools in a run ({n_pools_run})"
                    )
                pool_ids = pool_mean.index.to_numpy(dtype=np.int64)
                chunks = np.array_split(pool_ids, n_pool_bins)
                rows = [(run_id, int(pid), int(b)) for b, chunk in enumerate(chunks) for pid in chunk]
                return pd.DataFrame(rows, columns=["run", "pool_id", "pool_A_bin"])

            lo = float(np.min(pool_mean.to_numpy(dtype=float)))
            hi = float(np.max(pool_mean.to_numpy(dtype=float)))
            if hi <= lo:
                rows = [(run_id, int(pid), 0) for pid in pool_mean.index]
                return pd.DataFrame(rows, columns=["run", "pool_id", "pool_A_bin"])
            edges = np.linspace(lo, hi, n_pool_bins + 1)
            cats = pd.cut(pool_mean, bins=edges, include_lowest=True)
            codes = cats.cat.codes.to_numpy(dtype=np.int64)
            rows = [(run_id, int(pid), int(c)) for pid, c in zip(pool_mean.index, codes)]
            return pd.DataFrame(rows, columns=["run", "pool_id", "pool_A_bin"])

        maps = [pool_bin_map_for_run(sub) for _, sub in work.groupby("run", observed=True, sort=False)]
        pmap = pd.concat(maps, ignore_index=True)
        merged = work.merge(pmap, on=["run", "pool_id"], how="inner")
        per_run = (
            merged.groupby(["run", "pool_A_bin"], observed=True)
            .agg(
                x_mean=("A", "mean"),
                promotion_rate=("promoted", "mean"),
                n=("promoted", "size"),
                n_pools=("pool_id", "nunique"),
            )
            .reset_index()
        )
        summary = (
            per_run.groupby("pool_A_bin", observed=True)
            .agg(
                x_mean=("x_mean", "mean"),
                promotion_rate=("promotion_rate", "mean"),
                n=("n", "sum"),
                n_pools=("n_pools", "mean"),
            )
            .reset_index()
            .sort_values("x_mean", kind="mergesort")
            .reset_index(drop=True)
        )
        summary["x_col"] = label or "pool_mean_A_bins"
        return summary

    ABILITY_DISTRIBUTION_LABELS = {
        "A": r"$A_i \sim \mathrm{Uniform}(0, 1)$",
        "B": r"$A_i \sim \mathrm{clipped\ Normal}(0.5, 0.18)$",
        "C": r"$A_i \sim \mathrm{scaled\ Beta}(2, 5)$",
    }

    def ability_distribution_label(choice: str) -> str:
        """Human-readable ability distribution note for plot annotations."""
        return ABILITY_DISTRIBUTION_LABELS.get(choice, rf"$A_i$ distribution choice={choice!r}")

    POOL_SORTING_LABELS = {
        None: "Pools: none",
        "A": "Pools: random equal-sized",
        "B": "Pools: pure assortative",
        "C": "Pools: pure disassortative",
    }

    def pool_sorting_label(choice: str | None) -> str:
        """Human-readable pool sorting note for plot annotations."""
        return POOL_SORTING_LABELS.get(choice, f"Pools: sorting choice={choice!r}")

    WINNER_SELECTION_LABELS = {
        "A": "Weighted sample without replacement of K winners",
        "B": "Independent Bernoulli using min(normalized_weight×K, 1)",
        "C": "Deterministic top-K by promotion score",
    }

    def format_plot_na(value) -> str:
        if value is None:
            return "N/A"
        return str(value)

    def promotion_selection_label(score_mode: str, local_rank_weight: float | None = None) -> str:
        if score_mode == "ability":
            return "A_i"
        if score_mode == "global_rank":
            return "Global rank"
        if score_mode == "local_rank":
            return "Local rank"
        if score_mode == "local_rank_plus_ability":
            w = float(local_rank_weight) if local_rank_weight is not None else float("nan")
            return f"Additive, w={w:.3f}"
        return f"(unknown score_mode={score_mode!r})"

    def build_plot_meta(
        *,
        score_mode: str,
        x_binned: str,
        seed: int,
        winner_choice: str | None = None,
        ability_choice: str | None = None,
        pool_assignment_choice: str | None = None,
        n_pools: int | None = None,
        sorting_noise_sd: float | None = None,
        local_rank_weight: float | None = None,
        n_individuals: int | None = None,
        n_winners: int | None = None,
        n_runs: int | None = None,
        n_bins: int | None = None,
        caption_extra: str | None = None,
    ) -> dict:
        return {
            "ability_choice": ability_choice if ability_choice is not None else ABILITY_DISTRIBUTION_CHOICE,
            "score_mode": score_mode,
            "winner_choice": winner_choice if winner_choice is not None else WINNER_SELECTION_CHOICE,
            "pool_assignment_choice": pool_assignment_choice,
            "n_pools": n_pools,
            "sorting_noise_sd": sorting_noise_sd,
            "local_rank_weight": local_rank_weight,
            "n_individuals": int(n_individuals if n_individuals is not None else N_INDIVIDUALS),
            "n_winners": int(n_winners if n_winners is not None else N_WINNERS),
            "n_runs": int(n_runs if n_runs is not None else N_RUNS),
            "n_bins": int(n_bins if n_bins is not None else N_BINS),
            "seed": int(seed),
            "x_binned": x_binned,
            "caption_extra": caption_extra,
        }

    def annotate_simulation_context(ax, plot_meta: dict):
        """Standardized metadata under each panel: Selection + full simulation lineage."""
        ac = plot_meta["ability_choice"]
        sm = plot_meta["score_mode"]
        wc = plot_meta["winner_choice"]
        pool_c = plot_meta.get("pool_assignment_choice")
        n_pools = plot_meta.get("n_pools")
        snd = plot_meta.get("sorting_noise_sd")
        lr_w = plot_meta.get("local_rank_weight")
        n_i = plot_meta["n_individuals"]
        n_k = plot_meta["n_winners"]
        n_r = plot_meta["n_runs"]
        n_b = plot_meta["n_bins"]
        seed = plot_meta["seed"]
        xb = plot_meta["x_binned"]

        pool_line = pool_sorting_label(pool_c)
        if snd is not None and float(snd) > 0.0 and pool_c == "B":
            pool_line = "Pools: noisy assortative"
        elif snd is not None and float(snd) > 0.0 and pool_c == "C":
            pool_line = "Pools: noisy disassortative"

        draw_txt = WINNER_SELECTION_LABELS.get(wc, f"winner_choice={wc!r}")

        if sm == "local_rank_plus_ability" and lr_w is not None:
            w_line = f"{float(lr_w):.3f}"
        else:
            w_line = "N/A"

        if snd is None:
            noise_line = "N/A"
        else:
            noise_line = f"{float(snd):.4f}"

        sel_w = lr_w if sm == "local_rank_plus_ability" else None
        lines = [
            f"Selection: {promotion_selection_label(sm, sel_w)}",
            f"X-axis (binned): {xb}",
            f"Ability: {ability_distribution_label(ac)}",
            pool_line,
            f"Pool assignment: {format_plot_na(pool_c)}",
            f"N pools: {format_plot_na(n_pools)}",
            f"Sorting noise sd: {noise_line}",
            f"w (local-rank share in additive score): {w_line}",
            f"Draw: {draw_txt} ({wc})",
            f"N={n_i:,}; K={n_k}; runs={n_r:,}; bins={n_b}",
            f"Seed: {seed}",
        ]
        extra = plot_meta.get("caption_extra")
        if extra:
            lines.append(extra)
        ax.text(
            0.02,
            -0.10,
            "\n".join(lines),
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=9.0,
            color="0.28",
            clip_on=False,
        )
    def plot_bin_summary(
        summary: pd.DataFrame,
        title: str,
        xlabel: str,
        ax=None,
        *,
        show_simulation_context: bool = True,
        plot_meta: dict | None = None,
    ):
        """Plot binned promotion probability for one diagnostic relationship."""
        if ax is None:
            _, ax = plt.subplots(figsize=(FIG_PANEL_IN, FIG_PANEL_IN * 0.97))
        ax.plot(summary["x_mean"], summary["promotion_rate"], "o-", color="steelblue")
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Estimated promotion probability")
        ax.grid(True, alpha=0.3)
        if show_simulation_context:
            if plot_meta is None:
                raise ValueError("plot_meta is required when show_simulation_context=True")
            annotate_simulation_context(ax, plot_meta)
        return ax

    def plot_pool_A_bin_summary(
        summary: pd.DataFrame,
        title: str,
        ax=None,
        *,
        show_simulation_context: bool = True,
        plot_meta: dict | None = None,
    ):
        """Bar chart: Q1..QK along pool mean A (low to high), height = promotion rate."""
        if ax is None:
            _, ax = plt.subplots(figsize=(FIG_PANEL_IN, FIG_PANEL_IN * 0.97))
        xpos = np.arange(len(summary))
        ax.bar(xpos, summary["promotion_rate"], color="steelblue", alpha=0.88)
        ax.set_xticks(xpos)
        ax.set_xticklabels([f"Q{i + 1}" for i in xpos])
        ax.set_title(title)
        ax.set_xlabel(r"Pool mean $A_i$ bin (Q1 = lowest, QK = highest)")
        ax.set_ylabel("Estimated promotion probability")
        ax.grid(True, axis="y", alpha=0.3)
        if show_simulation_context:
            if plot_meta is None:
                raise ValueError("plot_meta is required when show_simulation_context=True")
            annotate_simulation_context(ax, plot_meta)
        return ax

    def maybe_save_figure(fig, filename_stem: str):
        """Save a figure only when SAVE_FIGURES is enabled in sim_config.py."""
        if not globals().get("SAVE_FIGURES", False):
            return None
        out_dir = Path("sports/outputs/simulation_figures")
        out_dir.mkdir(parents=True, exist_ok=True)
        out_path = out_dir / f"{filename_stem}.png"
        fig.savefig(out_path, dpi=200, bbox_inches="tight")
        print("Saved figure:", out_path)
        return out_path

    def show_figure(fig, filename_stem: str | None = None):
        """Display inline and close the figure; optionally save through maybe_save_figure."""
        if filename_stem is not None:
            maybe_save_figure(fig, filename_stem)
        display(fig)
        plt.close(fig)

    def simulate_population_rows(
        *,
        seed: int,
        n_runs: int,
        n: int,
        k: int,
        ability_choice: str,
        winner_choice: str,
        score_mode: str,
        include_random_pools: bool = False,
        n_pools: int | None = None,
        pool_assignment_choice: str = "A",
        sorting_noise_sd: float = 0.0,
        local_rank_weight: float = 0.5,
    ) -> pd.DataFrame:
        """Simulate repeated mock populations under one selection rule."""
        rng = np.random.default_rng(seed)
        rows: list[pd.DataFrame] = []
        for run in range(n_runs):
            A = generate_ability(rng, n, ability_choice)
            global_rank = rank_score(A)
            if include_random_pools:
                if n_pools is None:
                    raise ValueError("n_pools is required when include_random_pools=True")
                pool_id, sorting_signal = assign_pool_ids(
                    rng,
                    A,
                    n_pools,
                    choice=pool_assignment_choice,
                    sorting_noise_sd=sorting_noise_sd,
                )
                one = pd.DataFrame(
                    {
                        "run": run,
                        "person": np.arange(n),
                        "A": A,
                        "sorting_signal": sorting_signal,
                        "global_rank_score": global_rank,
                        "pool_id": pool_id,
                    }
                )
                one = add_pool_fields(one)
            else:
                one = pd.DataFrame({"run": run, "person": np.arange(n), "A": A, "global_rank_score": global_rank})

            if score_mode == "ability":
                weights = one["A"].to_numpy(dtype=float)
            elif score_mode == "global_rank":
                weights = one["global_rank_score"].to_numpy(dtype=float)
            elif score_mode == "local_rank":
                weights = one["local_rank_score"].to_numpy(dtype=float)
            elif score_mode == "local_rank_plus_ability":
                w = float(local_rank_weight)
                weights = w * one["local_rank_score"].to_numpy(dtype=float) + (1.0 - w) * one["A"].to_numpy(dtype=float)
            else:
                raise ValueError(f"Unknown score_mode: {score_mode!r}")

            one["promoted"] = choose_winners(rng, weights, k, winner_choice)
            rows.append(one)
        return pd.concat(rows, ignore_index=True)

    def checkpoint_list(checkpoints: list[int], max_runs: int) -> list[int]:
        """Keep positive unique checkpoints up to max_runs, always including max_runs."""
        vals = sorted({int(c) for c in checkpoints if 0 < int(c) <= int(max_runs)})
        if int(max_runs) not in vals:
            vals.append(int(max_runs))
        return vals

    def convergence_summaries(
        *,
        seed: int,
        checkpoints: list[int],
        n: int,
        k: int,
        ability_choice: str,
        winner_choice: str,
        score_mode: str,
        x_col: str,
        n_bins: int,
        include_random_pools: bool = False,
        n_pools: int | None = None,
        pool_assignment_choice: str = "A",
        sorting_noise_sd: float = 0.0,
        local_rank_weight: float = 0.5,
    ) -> dict[int, pd.DataFrame]:
        """Run one scenario at multiple iteration counts and return binned summaries."""
        out: dict[int, pd.DataFrame] = {}
        for n_runs in checkpoints:
            sim = simulate_population_rows(
                seed=seed,
                n_runs=n_runs,
                n=n,
                k=k,
                ability_choice=ability_choice,
                winner_choice=winner_choice,
                score_mode=score_mode,
                include_random_pools=include_random_pools,
                n_pools=n_pools,
                pool_assignment_choice=pool_assignment_choice,
                sorting_noise_sd=sorting_noise_sd,
                local_rank_weight=local_rank_weight,
            )
            out[n_runs] = summarize_by_bins(sim, x_col, n_bins, label=x_col)
        return out

    def plot_convergence_summaries(
        summaries: dict[int, pd.DataFrame],
        title: str,
        xlabel: str,
        ax=None,
        *,
        show_simulation_context: bool = True,
        plot_meta: dict | None = None,
    ):
        """Overlay checkpoint curves so noisy estimates visibly settle as runs increase."""
        if ax is None:
            _, ax = plt.subplots(figsize=(FIG_PANEL_IN * 1.65, FIG_PANEL_IN * 1.65))
        items = sorted(summaries.items())
        for idx, (n_runs, summary) in enumerate(items):
            alpha = 0.25 + 0.75 * ((idx + 1) / len(items))
            linewidth = 1.0 + 1.4 * ((idx + 1) / len(items))
            ax.plot(
                summary["x_mean"],
                summary["promotion_rate"],
                "o-",
                alpha=alpha,
                linewidth=linewidth,
                markersize=3.5,
                label=f"runs={n_runs}",
            )
        ax.set_title(title)
        ax.set_xlabel(xlabel)
        ax.set_ylabel("Estimated promotion probability")
        ax.grid(True, alpha=0.3)
        if show_simulation_context:
            if plot_meta is None:
                raise ValueError("plot_meta is required when show_simulation_context=True")
            annotate_simulation_context(ax, plot_meta)
        ax.legend(loc="upper right", fontsize=8)
        return ax
    print("CELL 1 ready: helpers loaded")
else:
    print("CELL 1 skipped (RUN_CELL1 = False)")


Reloaded simulation config from sim_config.py
N=1000, K=50, pools=50, runs=500, ability_choice='B', winner_choice='A', pool_choices=('A', 'B'), sorting_noise_sd=0.01, additive_w=0.0, convergence=False
CELL 1 ready: helpers loaded


### CELL 10 — Interactive EDA (one plot + knobs)

**Typical path:** **`sports/sim_config.py`** ↔ this cell. **Cell 1** once per kernel. **Rerun this cell** after saving config.

- **Binning · Individuals** — `pd.qcut` on the chosen x (`N_BINS` / *Person bins* when batch lock on).
- **Binning · Pools · equal count** — sort teams by pool mean \(A_i\), split into **K** near-equal pool groups (`N_POOL_AGG_BINS`). Requires `N_POOLS ≥ K`.
- **Binning · Pools · equal width** — within each run, **K** equal-width intervals on \([\min, \max]\) of pool mean \(A\); empty bands possible; Q order = increasing talent band.

Full reference: **`sports/documents/537_Manual.md`**.

`ipywidgets` required. Sliders update on release or **Run / refresh** (no preset).

In [8]:
# CELL 10 — Interactive playground (ipywidgets)

reload_sim_config()

if not RUN_CELL_PLAYGROUND:
    print("CELL 10 skipped (RUN_CELL_PLAYGROUND = False)")
else:
    try:
        import io

        import ipywidgets as widgets
        from IPython.display import Image, clear_output, display
    except ImportError:
        print("Install ipywidgets (e.g. pip install ipywidgets or conda install ipywidgets), then restart the kernel.")
    else:
        style = {"description_width": "160px"}
        lay = widgets.Layout(width="440px")

        w_noise = widgets.FloatSlider(
            value=float(SORTING_NOISE_SD),
            min=0.0,
            max=float(INTERACTIVE_SORTING_NOISE_MAX),
            step=0.005,
            description="Sorting noise sd",
            continuous_update=False,
            style=style,
            layout=lay,
        )
        w_additive = widgets.FloatSlider(
            value=float(ADDITIVE_LOCAL_RANK_WEIGHT),
            min=0.0,
            max=1.0,
            step=0.01,
            description="ADDITIVE w (local-rank share)",
            continuous_update=False,
            readout_format=".2f",
            style=style,
            layout=lay,
        )
        w_runs = widgets.IntSlider(
            value=int(INTERACTIVE_N_RUNS),
            min=20,
            max=500,
            step=10,
            description="Runs (playground)",
            continuous_update=False,
            style=style,
            layout=lay,
        )
        w_n = widgets.IntSlider(
            value=int(INTERACTIVE_N_INDIVIDUALS),
            min=200,
            max=2000,
            step=100,
            description="N individuals",
            continuous_update=False,
            style=style,
            layout=lay,
        )
        w_pool = widgets.Dropdown(
            options=[("Random equal pools", "A"), ("Assortative", "B"), ("Disassortative", "C")],
            value="B",
            description="Pool assignment",
            style=style,
            layout=lay,
        )
        w_score = widgets.Dropdown(
            options=[
                ("Local rank only", "local_rank"),
                ("w·local rank + (1-w)·A_i", "local_rank_plus_ability"),
            ],
            value="local_rank_plus_ability",
            description="Promotion score",
            style=style,
            layout=lay,
        )
        w_bins = widgets.IntSlider(
            value=int(INTERACTIVE_N_BINS),
            min=5,
            max=40,
            step=1,
            description="Person bins (qcut)",
            continuous_update=False,
            style=style,
            layout=lay,
        )
        w_pool_bins = widgets.IntSlider(
            value=int(globals().get("INTERACTIVE_N_POOL_AGG_BINS", 8)),
            min=2,
            max=40,
            step=1,
            description="Pool–talent bins (K)",
            continuous_update=False,
            style=style,
            layout=lay,
        )
        w_binning_mode = widgets.Dropdown(
            options=[
                ("Individuals: qcut on x", "individual_qcut"),
                ("Pools: equal pool count / bin (by mean A order)", "pool_equal_count"),
                ("Pools: equal width on mean A (per-run min–max)", "pool_equal_width"),
            ],
            value="individual_qcut",
            description="Binning",
            style=style,
            layout=widgets.Layout(width="560px"),
        )
        w_use_batch = widgets.Checkbox(
            value=bool(globals().get("INTERACTIVE_USE_MAIN_SCALES", False)),
            description="Batch lock: N, runs, N_BINS, N_POOL_AGG_BINS from sim_config",
            style=style,
            layout=widgets.Layout(width="520px"),
        )
        w_view = widgets.Dropdown(
            options=[
                ("With pools · local rank", "pool_local"),
                ("With pools · global rank", "pool_global"),
                ("With pools · ability A_i", "pool_A"),
                ("No pools · global rank (reference)", "nopool_global"),
            ],
            value="pool_local",
            description="Plot (x-axis)",
            style=style,
            layout=widgets.Layout(width="480px"),
        )
        w_ability = widgets.Dropdown(
            options=[
                ("A — Uniform(0,1)", "A"),
                ("B — clipped Normal(0.5, 0.18)", "B"),
                ("C — scaled Beta(2,5)", "C"),
            ],
            value=str(ABILITY_DISTRIBUTION_CHOICE),
            description="A_i distribution",
            style=style,
            layout=lay,
        )
        w_winner = widgets.Dropdown(
            options=[
                ("A — weighted K w/o replacement", "A"),
                ("B — independent Bernoulli", "B"),
                ("C — deterministic top-K", "C"),
            ],
            value=str(WINNER_SELECTION_CHOICE),
            description="Winner draw",
            style=style,
            layout=lay,
        )

        def _sync_playground_controls():
            locked = bool(w_use_batch.value)
            bmode = str(w_binning_mode.value)
            w_n.disabled = locked
            w_runs.disabled = locked
            w_bins.disabled = locked or bmode != "individual_qcut"
            w_pool_bins.disabled = locked or bmode not in ("pool_equal_count", "pool_equal_width")

        out = widgets.Output()
        _playground_ready = False

        def redraw(_=None):
            with out:
                clear_output(wait=True)
                noise = float(w_noise.value)
                wgt = float(w_additive.value)
                if w_use_batch.value:
                    n_play = int(N_INDIVIDUALS)
                    nruns = int(N_RUNS)
                    ibins = int(N_BINS)
                    pbins = int(N_POOL_AGG_BINS)
                else:
                    n_play = int(w_n.value)
                    nruns = int(w_runs.value)
                    ibins = int(w_bins.value)
                    pbins = int(w_pool_bins.value)
                pool = w_pool.value
                smode = w_score.value
                abil = str(w_ability.value)
                wchoice = str(w_winner.value)
                view = str(w_view.value)
                bmode = str(w_binning_mode.value)

                if smode == "local_rank":
                    lr_w = 0.5
                    meta_lr_w = None
                else:
                    lr_w = wgt
                    meta_lr_w = wgt

                w_additive.disabled = smode == "local_rank"
                _sync_playground_controls()

                pg_extra = (
                    "CELL 10 EDA (single plot). K and N_POOLS from sim_config.py. "
                    "FIG_PLAYGROUND_EDA_INCHES. Rerun cell after editing sim_config."
                )

                p_in = float(globals().get("FIG_PANEL_IN", 5.8))
                fig_wh = globals().get("FIG_PLAYGROUND_EDA_INCHES", (p_in * 1.4, p_in * 1.15))
                fig_w, fig_h = float(fig_wh[0]), float(fig_wh[1])
                fig, ax = plt.subplots(figsize=(fig_w, fig_h))

                seed_main = int(RANDOM_SEED + 7010)
                seed_nopool = int(RANDOM_SEED + 7000)

                if bmode in ("pool_equal_count", "pool_equal_width") and view == "nopool_global":
                    print(
                        "Pool-level binning needs pools. "
                        "Pick a pooled Plot (x-axis) or set Binning to individuals."
                    )
                    plt.close(fig)
                    return

                if bmode == "pool_equal_count" and int(N_POOLS) < pbins:
                    print(
                        f"Equal-count pool bins need N_POOLS ({int(N_POOLS)}) >= K ({pbins}). "
                        "Raise N_POOLS in sim_config or lower pool–talent bins."
                    )
                    plt.close(fig)
                    return

                use_pool_agg = bmode in ("pool_equal_count", "pool_equal_width")

                if view == "nopool_global":
                    df = simulate_population_rows(
                        seed=seed_nopool,
                        n_runs=nruns,
                        n=n_play,
                        k=N_WINNERS,
                        ability_choice=abil,
                        winner_choice=wchoice,
                        score_mode="global_rank",
                        include_random_pools=False,
                    )
                    xcol = "global_rank_score"
                    summ = summarize_by_bins(df, xcol, ibins, label=xcol)
                    summ = summ.sort_values("x_mean", kind="mergesort")
                    title = "No pools: promotion vs global rank"
                    xlab = "Global rank score"
                    meta = build_plot_meta(
                        score_mode="global_rank",
                        x_binned=f"global_rank_score: pd.qcut, {ibins} person bins",
                        seed=seed_nopool,
                        winner_choice=wchoice,
                        ability_choice=abil,
                        pool_assignment_choice=None,
                        n_pools=None,
                        sorting_noise_sd=None,
                        local_rank_weight=None,
                        n_individuals=n_play,
                        n_winners=N_WINNERS,
                        n_runs=nruns,
                        n_bins=ibins,
                        caption_extra=pg_extra,
                    )
                    plot_bin_summary(summ, title, xlab, ax=ax, plot_meta=meta)
                else:
                    df = simulate_population_rows(
                        seed=seed_main,
                        n_runs=nruns,
                        n=n_play,
                        k=N_WINNERS,
                        ability_choice=abil,
                        winner_choice=wchoice,
                        score_mode=smode,
                        include_random_pools=True,
                        n_pools=N_POOLS,
                        pool_assignment_choice=pool,
                        sorting_noise_sd=noise,
                        local_rank_weight=lr_w,
                    )
                    if use_pool_agg:
                        pool_m = "equal_count" if bmode == "pool_equal_count" else "equal_width"
                        try:
                            summ = summarize_by_pool_mean_A_bins(df, pbins, method=pool_m)
                        except ValueError as e:
                            print(e)
                            plt.close(fig)
                            return
                        per_pool = n_play // int(N_POOLS) if int(N_POOLS) else 0
                        title = "Promotion vs pool mean A_i (talent bins)"
                        if pool_m == "equal_count":
                            x_b = (
                                f"pools: equal COUNT / bin ({pbins}), sorted by mean A; "
                                f"~{float(summ['n_pools'].mean()):.1f} pools/bin; ~{per_pool} people/pool"
                            )
                        else:
                            x_b = (
                                f"pools: equal WIDTH on mean A ({pbins} intervals per run, [min,max] of pool means); "
                                f"~{per_pool} people/pool"
                            )
                        meta = build_plot_meta(
                            score_mode=smode,
                            x_binned=x_b,
                            seed=seed_main,
                            winner_choice=wchoice,
                            ability_choice=abil,
                            pool_assignment_choice=pool,
                            n_pools=N_POOLS,
                            sorting_noise_sd=noise,
                            local_rank_weight=meta_lr_w,
                            n_individuals=n_play,
                            n_winners=N_WINNERS,
                            n_runs=nruns,
                            n_bins=pbins,
                            caption_extra=pg_extra,
                        )
                        plot_pool_A_bin_summary(summ, title, ax=ax, plot_meta=meta)
                    else:
                        if view == "pool_local":
                            xcol = "local_rank_score"
                            xlab = "Local rank score"
                            title = "With pools: promotion vs local rank"
                            x_b = f"local_rank_score: pd.qcut, {ibins} person bins"
                        elif view == "pool_global":
                            xcol = "global_rank_score"
                            xlab = "Global rank score"
                            title = "With pools: promotion vs global rank"
                            x_b = f"global_rank_score: pd.qcut, {ibins} person bins"
                        else:
                            xcol = "A"
                            xlab = "Ability A_i"
                            title = "With pools: promotion vs ability A_i"
                            x_b = f"A: pd.qcut, {ibins} person bins"
                        summ = summarize_by_bins(df, xcol, ibins, label=xcol)
                        summ = summ.sort_values("x_mean", kind="mergesort")
                        meta = build_plot_meta(
                            score_mode=smode,
                            x_binned=x_b,
                            seed=seed_main,
                            winner_choice=wchoice,
                            ability_choice=abil,
                            pool_assignment_choice=pool,
                            n_pools=N_POOLS,
                            sorting_noise_sd=noise,
                            local_rank_weight=meta_lr_w,
                            n_individuals=n_play,
                            n_winners=N_WINNERS,
                            n_runs=nruns,
                            n_bins=ibins,
                            caption_extra=pg_extra,
                        )
                        plot_bin_summary(summ, title, xlab, ax=ax, plot_meta=meta)

                fig.subplots_adjust(bottom=0.40, top=0.94)
                buf = io.BytesIO()
                fig.savefig(
                    buf,
                    format="png",
                    dpi=144,
                    bbox_inches="tight",
                    facecolor=fig.get_facecolor(),
                    edgecolor="none",
                )
                plt.close(fig)
                buf.seek(0)
                display(Image(data=buf.getvalue()))
                pm = float(df["promoted"].mean())
                extra = ""
                if use_pool_agg:
                    extra = f"  x_mean(bin) [{float(summ['x_mean'].min()):.3f}, {float(summ['x_mean'].max()):.3f}]"
                print(
                    f"bmode={bmode} view={view}  mean promote={pm:.4f}  rows={len(df):,}  "
                    f"summary_len={len(summ)}  n min/max={int(summ['n'].min())}/{int(summ['n'].max())}"
                    f"{extra}"
                )


        btn = widgets.Button(description="Run / refresh")
        btn.on_click(redraw)

        def _on_change(change):
            if _playground_ready:
                redraw()

        def _on_binning_mode_change(change):
            if change.get("name") != "value":
                return
            _sync_playground_controls()
            if _playground_ready:
                redraw()

        def _on_use_batch_change(change):
            global _playground_ready
            if change.get("name") != "value":
                return
            if w_use_batch.value:
                pr = _playground_ready
                _playground_ready = False
                try:
                    w_n.value = int(N_INDIVIDUALS)
                    w_runs.value = int(N_RUNS)
                    w_bins.value = int(N_BINS)
                    w_pool_bins.value = int(N_POOL_AGG_BINS)
                finally:
                    _playground_ready = pr
            _sync_playground_controls()
            if _playground_ready:
                redraw()


        for w_ctrl in (
            w_noise,
            w_additive,
            w_runs,
            w_n,
            w_pool,
            w_score,
            w_bins,
            w_pool_bins,
            w_view,
            w_ability,
            w_winner,
        ):
            w_ctrl.observe(_on_change, names="value")
        w_binning_mode.observe(_on_binning_mode_change, names="value")
        w_use_batch.observe(_on_use_batch_change, names="value")

        ui = widgets.VBox(
            [
                widgets.HTML(
                    "<p><b>EDA:</b> one plot + caption under the axis. <b>Binning</b> = person <code>qcut</code> vs "
                    "pool <b>equal count</b> vs pool <b>equal width</b> on mean <code>A_i</code> (see "
                    "<code>sports/documents/537_Manual.md</code>). Batch lock = <code>N</code>, runs, <code>N_BINS</code>, "
                    "<code>N_POOL_AGG_BINS</code> from <code>sim_config.py</code>.</p>"
                ),
                w_use_batch,
                w_binning_mode,
                w_view,
                widgets.HBox([w_ability, w_winner]),
                widgets.HBox([w_noise, w_additive]),
                widgets.HBox([w_runs, w_n]),
                widgets.HBox([w_bins, w_pool_bins]),
                widgets.HBox([w_pool, w_score]),
                widgets.HBox([btn]),
                out,
            ]
        )
        display(ui)
        _playground_ready = True
        _sync_playground_controls()
        redraw()




Reloaded simulation config from sim_config.py
N=1000, K=50, pools=50, runs=500, ability_choice='B', winner_choice='A', pool_choices=('A', 'B'), sorting_noise_sd=0.01, additive_w=0.0, convergence=False
